# SIGPA-GA · Notebook 01 — Exploración de datos (Semana 21)

**DATOS DE DEMOSTRACIÓN.** Los datos de este cuaderno salen del
comando `exportar_dataset` (HU-166) hacia la carpeta `ML_EXPORT_DIR`,
**fuera del repositorio**. Ninguna cifra que aparezca aquí es un
resultado de la tesis: solo comprueba que el flujo de preparación
funciona.

## Qué hace
- Carga el CSV con la **misma puerta que usa el sistema** (`ml.carga.cargar`).
- Revisa faltantes, cobertura de clima y el catálogo de variables (anti-fuga).
- Mira la distribución del objetivo `rendimiento_kg_ha`.

## Qué NO hace
- **No entrena ningún modelo** (eso es Sprint 13).
- No escribe nada dentro del repositorio.

## Rutas

La carpeta de exportación se lee del `.env` de la raíz
(`ML_EXPORT_DIR=D:\MLDatasetSigpa`); si no existe, del entorno.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Raiz del repositorio (funciona desde notebooks/ o desde la raiz)
_candidatas = [Path.cwd(), Path.cwd().parent]
REPOSITORIO = next(
    (p for p in _candidatas if (p / "backend").is_dir() and (p / "notebooks").is_dir()),
    Path.cwd().parent,
)
sys.path.insert(0, str(REPOSITORIO / "backend"))


def valor_entorno(nombre, por_defecto):
    """Lee el .env de la raiz; si no, el entorno; si no, el defecto."""
    archivo = REPOSITORIO / ".env"
    if archivo.exists():
        for linea in archivo.read_text(encoding="utf-8").splitlines():
            if linea.startswith(nombre + "="):
                return linea.split("=", 1)[1].strip().strip("'\"")
    return os.environ.get(nombre, por_defecto)


# Los datos viven en ML_EXPORT_DIR (fuera de Git), nunca en el repositorio
RUTA_EXPORT = Path(valor_entorno("ML_EXPORT_DIR", r"D:\MLDatasetSigpa"))
print("Carpeta de exportacion:", RUTA_EXPORT)

## Carga con la puerta del sistema

`ml.carga.cargar` valida columnas y tipos **antes** de que nadie
use el archivo: faltantes o desconocidas rechazan con su lista.

In [ ]:
import pandas as pd

from ml import catalogo
from ml.carga import ErrorCarga, cargar

csvs = sorted(RUTA_EXPORT.glob("dataset_*.csv"))
if not csvs:
    raise SystemExit(
        "No hay dataset en %s. Ejecuta: manage.py exportar_dataset "
        "--usuario admin --origen-datos demostracion" % RUTA_EXPORT
    )
ruta_csv = csvs[-1]
datos = cargar(ruta_csv)
print("%s: %d filas x %d columnas" % (ruta_csv.name, datos.shape[0], datos.shape[1]))

In [ ]:
datos.head()

## Faltantes y cobertura de clima

In [ ]:
faltantes = datos.isna().mean().sort_values(ascending=False)
faltantes[faltantes > 0].round(3)

In [ ]:
ventana = datos["dias_ventana_clima"].replace(0, float("nan"))
cobertura = (datos["dias_con_clima"] / ventana).dropna()
print("Cobertura minima:", round(float(cobertura.min()), 3))
print("Campanas bajo el 80 %:", int((cobertura < catalogo.COBERTURA_CLIMA_MINIMA).sum()))

## Catálogo de variables (el contrato anti-fuga)

Lo que no está clasificado aquí no se usa; los identificadores,
metadatos, coberturas, seguimientos y `dias_ciclo` (fuga) jamás
entran en un conjunto de predictores.

In [ ]:
print("Sin clasificar:", catalogo.sin_clasificar(datos.columns) or "ninguna")
print("Conjunto 1 (plan):", len(catalogo.columnas_conjunto(catalogo.CONJUNTO_PLAN)), "predictores")
print("Conjunto 2 (plan + clima):", len(catalogo.columnas_conjunto(catalogo.CONJUNTO_CLIMA)), "predictores")
fuera_plan = catalogo.descartadas_por_faltantes(datos, catalogo.CONJUNTO_PLAN)
print("Fuera por faltantes (umbral 40 %):", fuera_plan or "ninguna")

## El objetivo: `rendimiento_kg_ha`

Se predice, nunca se imputa. Una fila sin objetivo se excluye en
la limpieza del notebook 02.

In [ ]:
datos["rendimiento_kg_ha"].describe()

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

ax = datos["rendimiento_kg_ha"].hist(bins=20, figsize=(6, 3))
ax.set_title("Rendimiento (kg/ha) - datos de demostracion")
ax.set_xlabel("kg/ha")
plt.tight_layout()
plt.show()

## Reglas de este cuaderno

- Se versiona **sin salidas**: los outputs se limpian antes de commitear
  y Jupyter nunca deja `.ipynb_checkpoints/` en Git (prueba 365).
- Para ejecutarlo y ver resultados:
  `jupyter nbconvert --to notebook --execute 01_exploracion.ipynb`
  desde el venv del backend.
- El siguiente paso es **02_preparacion.ipynb** (limpieza, pipeline,
  validación agrupada y línea base).